# Keras Model Quantization

- model.quantize("int8")
- model.quantize("int4")
- model.quantize("float8")
- model.quantize("gptq")

In [11]:
import numpy as np
from tensorflow.keras import Sequential, layers, ops, saving, models
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

In [12]:
X, y = make_regression(
    n_features=10,
    n_targets=1,
    n_samples=10000,
    n_informative=6,
    random_state=0
)

In [13]:
X.shape, y.shape

((10000, 10), (10000,))

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [25]:
# Create a random number generator.
model = Sequential([
    layers.InputLayer(shape=(10, )),
    layers.Dense(32, name='hidden1', activation='relu'),
    layers.Dense(128, name='hidden2', activation='relu'),
    layers.Dense(256, name='hidden3', activation='relu'),
    layers.Dense(512, name='hidden4', activation='relu'),
    layers.Dense(32, name='hidden5', activation='relu'),
    layers.Dense(1, name='target'),

])

# Compile and train briefly to materialize meaningful weights.
model.compile(optimizer="adam", loss="mse")

model.fit(X_train, y_train,
          epochs=10,
          batch_size=32,
          verbose=0,
          validation_data=(X_test, y_test))

baseline_rmse = root_mean_squared_error(
    y_test,
    model.predict(X_test, verbose=0)
)

In [ ]:
for quantization_method in ('int8', 'int4', None):
    if quantization_method is not None:

        quantized_model = models.clone_model(model)
        quantized_model.build(model.input_shape)
        quantized_model.set_weights(model.get_weights())

        quantized_model.quantize(quantization_method)
        y_pred = quantized_model.predict(X_test)
        RMSE = root_mean_squared_error(y_pred=y_pred, y_true=y_test)

        degradation_pct = (
                            (RMSE - baseline_rmse)
                            / baseline_rmse
                            * 100
                        )

        saving.save_model(quantized_model, f'rmse_{quantization_method}.keras', overwrite=True)
        # This is not workin` try use TensorFlowLite or ONNX

        print(f'Quantization method: {quantization_method}, RMSE: {RMSE}')
        print(f'Degradation %: {degradation_pct}')
    else:
        y_pred = model.predict(X_test)
        RMSE = root_mean_squared_error(y_pred=y_pred, y_true=y_test)
        print(f'Original method, score: {RMSE}')
        saving.save_model(quantized_model, f'rmse_orig.keras', overwrite=True)

## Saving and reloading a quantized model
You can use the standard Keras saving and loading APIs with quantized models. Quantization is preserved when saving to .keras and loading back.

In [ ]:
# Save the quantized model and reload to verify round-trip.
model.save("int8.keras")
int8_reloaded = saving.load_model("int8.keras")
y_int8_reloaded = int8_reloaded(x_eval)
roundtrip_mse = ops.mean(ops.square(y_int8 - y_int8_reloaded))
print("MSE (INT8 vs reloaded-INT8):", float(roundtrip_mse))


# Scikit-learn Model Quantization

- [Official documentation](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html?utm_source=chatgpt.com)

Scikit-learn itself does not provide native quantization APIs comparable to Keras/TensorFlow.

The common production path is:

ScikitLearnModel -> ONNX -> ONNX Runtime Quantization